SyntaxError: invalid syntax (3687545024.py, line 1)

In [6]:
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ----------------------------
# CONFIG
# ----------------------------
RBC_CARDS = [
    {
        "card_id": "rbc_avion_visa_infinite_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-avion-visa-infinite.html"
    },
    {
        "card_id": "rbc_ion_plus_visa_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/rewards/rbc-ion-plus-visa.html"
    },
    {
        "card_id": "rbc_world_elite_mastercard_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/westjet-rbc-world-elite-mastercard.html"
    },
    {
        "card_id": "rbc_cashback_mastercard_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/cash-back/rbc-cashback-mastercard.html"
    },
    {
        "card_id": "rbc_preferred_world_elite_mastercard_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/cash-back/rbc-preferred-world-elite-mastercard.html"
    },
    {
        "card_id": "rbc_visa_classic_low_rate_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/low-interest/rbc-visa-classic-low-rate.html"
    },
    {
        "card_id": "rbc_ion_visa_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/rewards/rbc-ion-visa.html"
    },
    {
        "card_id": "rbc_more_rewards_visa_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/rewards/more-rewards-rbc-visa.html"
    },
    {
        "card_id": "rbc_moi_visa_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/rewards/moi-rbc-visa.html"
    },
    {
        "card_id": "rbc_visa_platinum_avion_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-visa-platinum-avion.html"
    },
    {
        "card_id": "rbc_westjet_mastercard_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/westjet-rbc-mastercard.html"
    },
    {
        "card_id": "rbc_avion_visa_infinite_privilege_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-avion-visa-infinite-privilege.html"
    },
    {
        "card_id": "rbc_british_airways_visa_infinite_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-british-airways-visa-infinite.html"
    },
    {
        "card_id": "rbc_visa_platinum_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/no-fee/rbc-visa-platinum.html"
    },
    {
        "card_id": "rbc_us_dollar_visa_gold_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-us-dollar-visa-gold.html"
    },
    {
        "card_id": "rbc_rateadvantage_visa_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/low-interest/rbc-rateadvantage-visa.html"
    },
    {
        "card_id": "rbc_more_rewards_visa_infinite_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/rewards/more-rewards-rbc-visa-infinite.html"
    },
]

DATA_DIR = Path(r"C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_rbc.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_rbc.jsonl"


# ----------------------------
# BROWSER
# ----------------------------
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1700,3000")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


# ----------------------------
# HELPERS
# ----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    if not text:
        return ""
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def normalize_money(value: str) -> str:
    if not value:
        return ""
    return value.replace("$", "").replace(",", "").strip()


def normalize_card_name(name: str) -> str:
    if not name:
        return ""
    name = clean_text(name)
    name = name.replace("*", "")
    name = re.sub(r"\s+", " ", name).strip()
    return name


def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1800):
    if not text:
        return ""

    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        if not kw:
            continue
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            if not kw:
                continue
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])


def fetch_page_text(driver, url: str, wait_sec: int = 8):
    driver.get(url)
    time.sleep(wait_sec)

    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))

    return html, soup, page_text


# ----------------------------
# RBC CARD NAME FIX
# ----------------------------
def rbc_card_name_from_card_id(card_id: str):
    mapping = {
        "rbc_avion_visa_infinite_ca": "RBC Avion Visa Infinite",
        "rbc_ion_plus_visa_ca": "RBC ION+ Visa",
        "rbc_world_elite_mastercard_ca": "WestJet RBC World Elite Mastercard",
        "rbc_cashback_mastercard_ca": "RBC Cashback Mastercard",
        "rbc_preferred_world_elite_mastercard_ca": "RBC Preferred World Elite Mastercard",
        "rbc_visa_classic_low_rate_ca": "RBC Visa Classic Low Rate Option",
        "rbc_ion_visa_ca": "RBC ION Visa",
        "rbc_more_rewards_visa_ca": "More Rewards RBC Visa",
        "rbc_moi_visa_ca": "Moi RBC Visa",
        "rbc_visa_platinum_avion_ca": "RBC Visa Platinum Avion",
        "rbc_westjet_mastercard_ca": "WestJet RBC Mastercard",
        "rbc_avion_visa_infinite_privilege_ca": "RBC Avion Visa Infinite Privilege",
        "rbc_british_airways_visa_infinite_ca": "RBC British Airways Visa Infinite",
        "rbc_visa_platinum_ca": "RBC Visa Platinum",
        "rbc_us_dollar_visa_gold_ca": "RBC U.S. Dollar Visa Gold",
        "rbc_rateadvantage_visa_ca": "RBC RateAdvantage Visa",
        "rbc_more_rewards_visa_infinite_ca": "More Rewards RBC Visa Infinite",
    }
    return mapping.get(card_id, card_id.replace("_", " ").title())


def rbc_card_name_from_url(url: str):
    mapping = {
        "rbc-avion-visa-infinite": "RBC Avion Visa Infinite",
        "rbc-ion-plus-visa": "RBC ION+ Visa",
        "westjet-rbc-world-elite-mastercard": "WestJet RBC World Elite Mastercard",
        "rbc-cashback-mastercard": "RBC Cashback Mastercard",
        "rbc-preferred-world-elite-mastercard": "RBC Preferred World Elite Mastercard",
        "rbc-visa-classic-low-rate": "RBC Visa Classic Low Rate Option",
        "rbc-ion-visa": "RBC ION Visa",
        "more-rewards-rbc-visa": "More Rewards RBC Visa",
        "moi-rbc-visa": "Moi RBC Visa",
        "rbc-visa-platinum-avion": "RBC Visa Platinum Avion",
        "westjet-rbc-mastercard": "WestJet RBC Mastercard",
        "rbc-avion-visa-infinite-privilege": "RBC Avion Visa Infinite Privilege",
        "rbc-british-airways-visa-infinite": "RBC British Airways Visa Infinite",
        "rbc-visa-platinum": "RBC Visa Platinum",
        "rbc-us-dollar-visa-gold": "RBC U.S. Dollar Visa Gold",
        "rbc-rateadvantage-visa": "RBC RateAdvantage Visa",
        "more-rewards-rbc-visa-infinite": "More Rewards RBC Visa Infinite",
    }

    for slug, title in mapping.items():
        if slug in url:
            return title
    return ""


def extract_generic_rbc_card_name(driver, soup, page_text: str, card_cfg: dict):
    card_id = card_cfg["card_id"]
    url = card_cfg["url"]

    # 1. URL-based mapping
    mapped_name = rbc_card_name_from_url(url)
    if mapped_name:
        return mapped_name

    # 2. Title tag
    if soup.title and soup.title.text:
        title = normalize_card_name(soup.title.text)
        title = title.split("|")[0].strip()

        if any(x in title.lower() for x in ["rbc", "westjet", "more rewards", "moi"]):
            return title

    # 3. Visible headings
    candidate_selectors = [
        "h1",
        ".hero-banner__title",
        ".cmp-title__text",
        ".product-name",
        ".card-title",
        ".page-title",
    ]

    for selector in candidate_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                txt = normalize_card_name(elem.text)
                if not txt:
                    continue
                txt_lower = txt.lower()
                if any(x in txt_lower for x in ["rbc", "westjet", "more rewards", "moi"]):
                    if len(txt) < 150:
                        return txt
        except Exception:
            pass

    # 4. Regex fallback
    patterns = [
        r"(RBC Avion Visa Infinite Privilege)",
        r"(RBC Avion Visa Infinite)",
        r"(RBC Visa Platinum Avion)",
        r"(RBC British Airways Visa Infinite)",
        r"(RBC ION\+ Visa)",
        r"(RBC ION Visa)",
        r"(RBC Cashback Mastercard)",
        r"(RBC Preferred World Elite Mastercard)",
        r"(WestJet RBC World Elite Mastercard)",
        r"(WestJet RBC Mastercard)",
        r"(RBC Visa Platinum)",
        r"(RBC U\.S\. Dollar Visa Gold)",
        r"(RBC RateAdvantage Visa)",
        r"(RBC Visa Classic Low Rate Option)",
        r"(More Rewards RBC Visa Infinite)",
        r"(More Rewards RBC Visa)",
        r"(Moi RBC Visa)",
    ]
    for pattern in patterns:
        found = search_first([pattern], page_text)
        if found:
            return normalize_card_name(found)

    # 5. Final fallback
    return rbc_card_name_from_card_id(card_id)


# ----------------------------
# GENERIC FIELD EXTRACTION
# ----------------------------
def extract_rbc_annual_fee(page_text: str):
    fee = search_first(
        [
            r"Annual Fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual Fee\s+\$([0-9]+(?:\.[0-9]{1,2})?)",
        ],
        page_text
    )
    return normalize_money(fee)


def extract_rbc_additional_card_fee(page_text: str):
    fee = search_first(
        [
            r"Additional Card[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Additional Cardholder[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Additional Card\s+\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_rbc_purchase_rate(page_text: str):
    return search_first(
        [
            r"Purchase Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Interest[:\s]*Purchases[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Purchase Rate\s+([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )


def extract_rbc_cash_advance_rate(page_text: str):
    return search_first(
        [
            r"Cash Advance Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%[\*]?)",
            r"Interest[:\s]*Cash Advances[:\s]*([0-9]+(?:\.[0-9]{1,2})?%[\*]?)",
            r"Cash Advance Rate\s+([0-9]+(?:\.[0-9]{1,2})?%[\*]?)"
        ],
        page_text
    )


def extract_rbc_income_requirement(page_text: str):
    patterns = [
        r"(A minimum personal income of \$[0-9,]+ or a minimum household income of \$[0-9,]+ is required for this card\.)",
        r"(\$[0-9,]+\s*personal income or \$[0-9,]+\s*household income)",
        r"(\$[0-9,]+\s*personal or \$[0-9,]+\s*household income)"
    ]
    return search_first(patterns, page_text)


def extract_rbc_welcome_bonus(page_text: str):
    patterns = [
        r"(Get up to [^\.\n]*?points[^\.\n]*\.)",
        r"(Get up to [^\.\n]*?WestJet dollars[^\.\n]*\.)",
        r"(Get up to [^\.\n]*?cash back[^\.\n]*\.)",
        r"(Welcome [^\.\n]*\.)",
        r"(Special Offer [^\.\n]*\.)",
    ]
    return search_first(patterns, page_text)


def extract_rbc_rewards_summary(page_text: str):
    patterns = [
        r"(Earn [0-9\.]+ .*? on .*? purchases[^\.;]*)",
        r"(Earn [0-9\.]+ .*? on all other eligible purchases[^\.;]*)",
        r"(Earn up to [0-9\.]+% cash back[^\.;]*)",
        r"(Earn [0-9\.]+ WestJet dollars[^\.;]*)",
    ]

    found_parts = []
    for pattern in patterns:
        matches = re.findall(pattern, page_text, re.IGNORECASE)
        for m in matches:
            m = clean_text(m)
            if m not in found_parts:
                found_parts.append(m)

    return clean_text(" | ".join(found_parts[:4]))


def detect_rbc_network(page_text: str, card_name: str):
    text = f"{card_name} {page_text}".lower()
    if "mastercard" in text:
        return "Mastercard"
    if "visa" in text:
        return "Visa"
    return ""


def infer_best_for(page_text: str, card_name: str):
    text = f"{card_name} {page_text}".lower()
    tags = []

    if any(x in text for x in ["travel", "flight", "airline", "hotel", "car rentals"]):
        tags.append("travel")
    if "avion" in text:
        tags.append("flexible_travel_rewards")
    if "westjet" in text:
        tags.append("airline_rewards")
    if "cash back" in text or "cashback" in text:
        tags.append("cashback")
    if "gas" in text or "fuel" in text or "petro-canada" in text:
        tags.append("gas")
    if "doordash" in text:
        tags.append("food_delivery")
    if "hertz" in text or "car rentals" in text:
        tags.append("car_rental")
    if "visa infinite" in text or "world elite" in text:
        tags.append("premium_perks")
    if "low rate" in text or "rateadvantage" in text:
        tags.append("low_interest")
    if "rewards" in text:
        tags.append("rewards")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)


def infer_one_liner(page_text: str, card_name: str):
    text = f"{card_name} {page_text}".lower()

    if "avion" in text:
        return "Canadian travel rewards card for flexible flight redemptions, travel purchases, and premium perks."
    if "westjet" in text:
        return "Airline rewards card for WestJet travel benefits and travel redemptions."
    if "cashback" in text or "cash back" in text:
        return "Canadian cash back credit card for everyday spending."
    if "low rate" in text or "rateadvantage" in text:
        return "Low-interest credit card designed to reduce borrowing costs."
    if "ion" in text:
        return "Rewards card for everyday spending with RBC Avion-style earning options."

    return "Canadian RBC credit card with rewards and cardholder benefits."


# ----------------------------
# MAIN SCRAPER
# ----------------------------
def scrape_rbc_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_generic_rbc_card_name(driver, soup, page_text, card_cfg)
    issuer = "RBC"
    country = "Canada"
    network = detect_rbc_network(page_text, card_name)
    card_type = "Credit Card"

    annual_fee = extract_rbc_annual_fee(page_text)
    additional_card_fee = extract_rbc_additional_card_fee(page_text)
    purchase_rate = extract_rbc_purchase_rate(page_text)
    cash_advance_rate = extract_rbc_cash_advance_rate(page_text)

    welcome_bonus_summary = extract_rbc_welcome_bonus(page_text)
    eligibility_summary = extract_rbc_income_requirement(page_text)

    best_for = infer_best_for(page_text, card_name)
    one_liner = infer_one_liner(page_text, card_name)
    rewards_summary = extract_rbc_rewards_summary(page_text)

    overview_text = extract_section(
        page_text,
        start_keywords=[card_name, "Special Offer", "Card Highlights", "Experience"],
        end_keywords=["How to Earn", "How to Redeem", "Card Highlights", "Benefits"],
        max_chars=1200
    )

    highlights_text = extract_section(
        page_text,
        start_keywords=["Card Highlights", "Benefits", "Why Choose"],
        end_keywords=["How to Earn", "How to Redeem", "More Details", "Additional Benefits"],
        max_chars=1500
    )

    rewards_text = extract_section(
        page_text,
        start_keywords=["How to Earn", "Earn", "Rewards"],
        end_keywords=["How to Redeem", "Perks", "Benefits", "Insurance"],
        max_chars=1500
    )

    redemption_text = extract_section(
        page_text,
        start_keywords=["How to Redeem", "Redeem", "Use your"],
        end_keywords=["Perks", "Benefits", "Insurance", "More Details"],
        max_chars=1500
    )

    partner_offers_text = extract_section(
        page_text,
        start_keywords=["Exclusive Offers", "Partner rewards", "Offers From Brands"],
        end_keywords=["Visa Infinite", "World Elite", "Why Choose", "More Details"],
        max_chars=1500
    )

    premium_benefits_text = extract_section(
        page_text,
        start_keywords=["Visa Infinite", "World Elite", "Premium Benefits"],
        end_keywords=["Why Choose", "More Details", "Insurance", "Optional Add-On"],
        max_chars=1500
    )

    insurance_text = extract_section(
        page_text,
        start_keywords=["Insurance", "Extensive Insurance Coverage", "Get Extensive Insurance"],
        end_keywords=["Even More Ways", "Optional Add-On", "Why Choose", "More Details"],
        max_chars=1500
    )

    fees_text = clean_text(
        f"Annual fee: ${annual_fee}; Additional card fee: ${additional_card_fee}; "
        f"Purchase rate: {purchase_rate}; Cash advance rate: {cash_advance_rate}"
    )

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": "",
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,
        "additional_card_fee": additional_card_fee,
        "purchase_rate": purchase_rate,
        "cash_advance_rate": cash_advance_rate,
        "rewards_summary": rewards_summary
    }

    snippets = [
        {
            "chunk_id": f"{card_id}_overview",
            "card_id": card_id,
            "section": "overview",
            "text": overview_text
        },
        {
            "chunk_id": f"{card_id}_highlights",
            "card_id": card_id,
            "section": "highlights",
            "text": highlights_text
        },
        {
            "chunk_id": f"{card_id}_rewards",
            "card_id": card_id,
            "section": "rewards",
            "text": rewards_text
        },
        {
            "chunk_id": f"{card_id}_welcome_bonus",
            "card_id": card_id,
            "section": "welcome_bonus",
            "text": welcome_bonus_summary
        },
        {
            "chunk_id": f"{card_id}_redemption",
            "card_id": card_id,
            "section": "redemption",
            "text": redemption_text
        },
        {
            "chunk_id": f"{card_id}_partner_offers",
            "card_id": card_id,
            "section": "partner_offers",
            "text": partner_offers_text
        },
        {
            "chunk_id": f"{card_id}_premium_benefits",
            "card_id": card_id,
            "section": "premium_benefits",
            "text": premium_benefits_text
        },
        {
            "chunk_id": f"{card_id}_insurance",
            "card_id": card_id,
            "section": "insurance",
            "text": insurance_text
        },
        {
            "chunk_id": f"{card_id}_fees",
            "card_id": card_id,
            "section": "fees",
            "text": fees_text
        },
        {
            "chunk_id": f"{card_id}_eligibility",
            "card_id": card_id,
            "section": "eligibility",
            "text": eligibility_summary
        }
    ]

    snippets = [s for s in snippets if s["text"]]

    return card_record, snippets


# ----------------------------
# RUN
# ----------------------------
def main():
    driver = get_driver(headless=True)

    all_cards = []
    all_snippets = []

    try:
        for card in RBC_CARDS:
            print(f"Scraping: {card['card_id']}")
            card_record, snippets = scrape_rbc_card(driver, card)

            print("Extracted card name:", card_record["card_name"])

            all_cards.append(card_record)
            all_snippets.extend(snippets)
            time.sleep(2)
    finally:
        driver.quit()

    cards_df = pd.DataFrame(all_cards)
    cards_df.to_csv(CSV_PATH, index=False)

    with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
        for item in all_snippets:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved CSV to:", CSV_PATH)
    print("Saved snippets to:", SNIPPETS_PATH)
    print(cards_df[["card_id", "card_name", "annual_fee", "network"]].T)

    print("\nSnippet preview:")
    for s in all_snippets[:15]:
        print(f"- {s['section']}: {s['text'][:180]}...")


if __name__ == "__main__":
    main()

Scraping: rbc_avion_visa_infinite_ca
Extracted card name: RBC Avion Visa Infinite
Scraping: rbc_ion_plus_visa_ca
Extracted card name: RBC ION+ Visa
Scraping: rbc_world_elite_mastercard_ca
Extracted card name: WestJet RBC World Elite Mastercard
Scraping: rbc_cashback_mastercard_ca
Extracted card name: RBC Cashback Mastercard
Scraping: rbc_preferred_world_elite_mastercard_ca
Extracted card name: RBC Preferred World Elite Mastercard
Scraping: rbc_visa_classic_low_rate_ca
Extracted card name: RBC Visa Classic Low Rate Option
Scraping: rbc_ion_visa_ca
Extracted card name: RBC ION Visa
Scraping: rbc_more_rewards_visa_ca
Extracted card name: More Rewards RBC Visa
Scraping: rbc_moi_visa_ca
Extracted card name: Moi RBC Visa
Scraping: rbc_visa_platinum_avion_ca
Extracted card name: RBC Visa Platinum Avion
Scraping: rbc_westjet_mastercard_ca
Extracted card name: WestJet RBC Mastercard
Scraping: rbc_avion_visa_infinite_privilege_ca
Extracted card name: RBC Avion Visa Infinite
Scraping: rbc_british